# Segmentação de pluma de fumaça com SAM

Continuação do notebook inicial da atividade, com a mesma stack (Ultralytics). O que mudou foi a
forma de acionar o SAM e a forma de avaliar o resultado.

## O que estava quebrado no notebook inicial

| Célula | O que acontecia | Causa |
|---|---|---|
| 13 | `FileNotFoundError: 'sam_b.pt'` | `SAM3SemanticPredictor` espera pesos do SAM 3, cujo download deu 404. `sam_b.pt` não aceita prompt de texto. |
| 14 | `(no detections)` e nenhuma máscara gerada | o detector `YOLOv8m-smoke-detection` não encontrou a pluma |
| 12 | "4 cars, 1 truck" | `yolo26n` é COCO, não tem classe de fumaça |
| 7 | `copyfile("pluma.jpg", "pluma.png")` | só renomeia, o arquivo continua JPEG |

Aqui o SAM é acionado por prompt geométrico (caixa e pontos), com `SAM2Predictor`, e `set_image()` é
chamado uma vez só.

Aviso sobre a célula 4 do notebook inicial: ela tem uma chave da API do CDS/Copernicus escrita no
código. Em repositório público isso é uma chave vazada. Revogue essa e gere outra no painel do CDS, e
leia a nova de variável de ambiente (`os.environ["CDSAPI_KEY"]`). Este notebook não tem a chave nem a
parte de ERA5/netCDF, que é outra frente do projeto.

## Imagem

A imagem do notebook inicial (`MyDrive/pluma.jpg`) não está mais disponível. A análise usa uma foto
pública de domínio público (incêndio no sul da Califórnia, outubro de 2007, Wikimedia Commons), que
tem os três distratores citados no enunciado ao mesmo tempo: céu azul, casas e uma faixa de haze
esbranquiçada quase indistinguível da base da pluma. Há também uma placa de trânsito ocluindo parte
da pluma.

Para trocar de imagem, mude `CFG["IMAGEM"]` e reescreva `PROMPTS` na Etapa 2 com as coordenadas da
sua pluma. A Etapa 1 imprime a imagem com eixos em pixel: divida pela largura e pela altura do título
para obter a fração.

Para rodar, use `Executar tudo` no menu `Ambiente de execução`. Com GPU usa `sam2_b.pt`, sem GPU usa `sam2_t.pt`.

In [ ]:
# ============================================================
# ETAPA 1 — Instalação, imagem e prompts
# ============================================================
!pip install -q ultralytics

import os, urllib.request
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

np.random.seed(42); torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.titlesize": 10, "axes.titleweight": "bold", "font.size": 9})
os.makedirs("saida", exist_ok=True)

CFG = {"IMAGEM": "socal"}          # "socal", "upload", ou um caminho/URL
URL = ("https://commons.wikimedia.org/wiki/Special:FilePath/"
       "Smoke%20plume%20from%20SoCal%20wildfire%20October%202007.jpg?width=1280")
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
MODELO = "sam2_b.pt" if DISPOSITIVO == "cuda" else "sam2_t.pt"   # Ultralytics baixa sozinho

if CFG["IMAGEM"] == "socal":
    CAMINHO = "pluma_socal.jpg"
    if not os.path.exists(CAMINHO):
        req = urllib.request.Request(URL, headers={"User-Agent": "colab-sam/1.0"})
        open(CAMINHO, "wb").write(urllib.request.urlopen(req).read())
elif CFG["IMAGEM"] == "upload":
    from google.colab import files
    CAMINHO = list(files.upload().keys())[0]
else:
    CAMINHO = CFG["IMAGEM"]

IMG = cv2.cvtColor(cv2.imread(CAMINHO), cv2.COLOR_BGR2RGB)  # cv2 lê pelo conteúdo, não pela extensão
ALT, LARG = IMG.shape[:2]
print(f"imagem {LARG}x{ALT} | dispositivo={DISPOSITIVO} | modelo={MODELO}")

# Prompts em coordenadas fracionárias (x/largura, y/altura), calibrados nesta imagem.
# Fração em vez de pixel: a mesma lista vale em qualquer resolução.
PROMPTS = {
    "caixa": (0.289, 0.266, 0.999, 0.625),
    "positivos": [(0.336, 0.479), (0.484, 0.438), (0.625, 0.385), (0.742, 0.344), (0.320, 0.531)],
    "negativos": [(0.117, 0.156), (0.547, 0.156), (0.781, 0.125),   # céu azul
                  (0.234, 0.583), (0.117, 0.625), (0.570, 0.635),   # encosta e telhados
                  (0.469, 0.833), (0.883, 0.271)],                  # asfalto e a placa que oclui
}
ROIS_FRAC = {"ceu": (0.05, 0.05, 0.30, 0.20), "nuvem_haze": (0.60, 0.50, 0.78, 0.56),
             "edificacao": (0.02, 0.56, 0.22, 0.66)}
# Referência traçada à mão sobre a imagem. É aproximada: ver o aviso da Etapa 4.
REF_FRAC = [(0.305, 0.552), (0.316, 0.490), (0.328, 0.448), (0.352, 0.425), (0.391, 0.420),
            (0.426, 0.442), (0.438, 0.408), (0.469, 0.375), (0.516, 0.359), (0.563, 0.359),
            (0.609, 0.344), (0.648, 0.313), (0.688, 0.297), (0.727, 0.286), (0.773, 0.292),
            (0.820, 0.302), (0.875, 0.302), (0.938, 0.297), (0.996, 0.302), (0.998, 0.392),
            (0.930, 0.460), (0.855, 0.500), (0.781, 0.530), (0.703, 0.545), (0.625, 0.540),
            (0.547, 0.526), (0.484, 0.516), (0.430, 0.521), (0.359, 0.536)]


def para_px(pts):
    p = np.asarray(pts, float).reshape(-1, 2)
    return np.stack([p[:, 0] * LARG, p[:, 1] * ALT], axis=1)


def caixa_px(c):
    return np.array([c[0] * LARG, c[1] * ALT, c[2] * LARG, c[3] * ALT], float)


def roi_mascara(c):
    x0, y0, x1, y1 = caixa_px(c).astype(int)
    m = np.zeros((ALT, LARG), bool); m[max(y0, 0):y1, max(x0, 0):x1] = True
    return m


calibrado = CFG["IMAGEM"] == "socal"
ROIS = {n: roi_mascara(c) for n, c in ROIS_FRAC.items()} if calibrado else {}
if calibrado:
    REFERENCIA = np.zeros((ALT, LARG), np.uint8)
    cv2.fillPoly(REFERENCIA, [para_px(REF_FRAC).astype(np.int32)], 1)
    REFERENCIA = REFERENCIA.astype(bool)
else:
    REFERENCIA = None
    print("Imagem trocada: os prompts e a referência calibrados não valem mais. Reescreva PROMPTS.")

fig, ax = plt.subplots(figsize=(10, 10 * ALT / LARG))
ax.imshow(IMG); ax.grid(color="yellow", lw=.4, alpha=.6)
ax.set_xticks(range(0, LARG, 100)); ax.set_yticks(range(0, ALT, 100))
ax.tick_params(labelsize=6)
if REFERENCIA is not None:
    ax.contour(REFERENCIA, [0.5], colors="deepskyblue", linewidths=2)
ax.set_title(f"{CFG['IMAGEM']} ({LARG}x{ALT} px). Azul: referência traçada à mão")
fig.tight_layout(); fig.savefig("saida/01_grade.png", bbox_inches="tight"); plt.show()

---
# Etapa 2: três configurações de prompt

| | Prompt | Hipótese testada |
|---|---|---|
| A | só a caixa | o retângulo basta? Ele contém muito céu junto da pluma. |
| B | só pontos positivos | sem limite espacial, até onde a região espalha por semelhança? |
| C | caixa + 5 positivos + 8 negativos | declarar o que não é pluma resolve a ambiguidade? |

O SAM é agnóstico a classe: ele não tem o conceito de fumaça, devolve a região coerente que o prompt
indica. Por isso o caminho do notebook inicial, deixar um detector achar a fumaça, falhou duas vezes:
o detector não detectou nada e o prompt de texto exigia um modelo indisponível. Com prompt geométrico
o controle volta para quem está segmentando.

O ponto negativo é o elemento que faz diferença aqui, porque céu claro, haze e telhado estão dentro
da caixa. Nenhum ajuste do retângulo exclui um distrator que mora no interior dele; um ponto rotulado
como fundo exclui.

Detalhe de API que custou um erro em tempo de execução: o SAM 2 da Ultralytics funde caixa e pontos
num único tensor de prompts, com a caixa virando `(1, 2, 2)`. Pontos passados em `(N, 2)` são lidos
como N objetos de um ponto cada, e o `torch.cat` falha com `Sizes of tensors must match except in
dimension 1`. Por isso a função `prever` envia os pontos agrupados em `(1, N, 2)` e os rótulos em
`(1, N)`: assim os N pontos pertencem ao mesmo objeto da caixa.

In [ ]:
# ============================================================
# ETAPA 2 — Predição
# ============================================================
from ultralytics.models.sam import SAM2Predictor

predictor = SAM2Predictor(overrides=dict(task="segment", mode="predict", model=MODELO,
                                         save=False, verbose=False, conf=0.25))
predictor.set_image(CAMINHO)       # custo alto: codificar a imagem. Feito uma vez só.


def prever(caixa=None, pontos=None, rotulos=None):
    """Chama o SAM com prompt geométrico e devolve uma máscara booleana HxW."""
    kw = {}
    if caixa is not None:
        kw["bboxes"] = caixa_px(caixa)[None, :]
    if pontos:
        # Os pontos vão agrupados num eixo extra, (1, N, 2), e os rótulos em (1, N).
        # O SAM 2 da Ultralytics funde caixa e pontos num tensor só, com a caixa virando
        # (1, 2, 2); pontos em (N, 2) seriam lidos como N objetos de um ponto cada e o
        # torch.cat falharia. Agrupados, os N pontos pertencem ao mesmo objeto da caixa.
        kw["points"] = para_px(pontos)[None]
        kw["labels"] = np.asarray(rotulos, int)[None]
    r = predictor(**kw)[0]
    if r.masks is None or len(r.masks.data) == 0:
        return np.zeros((ALT, LARG), bool)
    m = r.masks.data.cpu().numpy().any(0).astype(np.uint8)
    if m.shape != (ALT, LARG):
        m = cv2.resize(m, (LARG, ALT), interpolation=cv2.INTER_NEAREST)
    return m.astype(bool)


n_pos, n_neg = len(PROMPTS["positivos"]), len(PROMPTS["negativos"])
CONFIGS = {
    "A: só caixa": dict(caixa=PROMPTS["caixa"]),
    "B: só pontos (+)": dict(pontos=PROMPTS["positivos"], rotulos=np.ones(n_pos, int)),
    "C: caixa + (+) + (-)": dict(caixa=PROMPTS["caixa"],
                                 pontos=list(PROMPTS["positivos"]) + list(PROMPTS["negativos"]),
                                 rotulos=np.r_[np.ones(n_pos, int), np.zeros(n_neg, int)]),
}
MASCARAS = {nome: prever(**cfg) for nome, cfg in CONFIGS.items()}
for nome, m in MASCARAS.items():
    print(f"{nome:24s} área = {100*m.mean():5.2f}% da imagem")

---
# Etapa 3: apresentação da máscara

Original com os prompts desenhados, máscara binária e máscara sobreposta, lado a lado, para cada
configuração. Os prompts aparecem sobre a original porque sem isso não dá para saber se o erro veio
do modelo ou de um ponto mal colocado. A sobreposição usa transparência com contorno, já que
preenchimento opaco esconde a borda, que é justamente onde está o erro interessante. O painel binário
traz o percentual de área para dar escala.

In [ ]:
# ============================================================
# ETAPA 3 — Figuras lado a lado
# ============================================================

def sobrepor(img, mask, alpha=0.45):
    out = img.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array([255, 60, 60], np.float32)
    out = out.astype(np.uint8)
    cont, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cont, -1, (255, 230, 0), 2)
    return out


fig, axs = plt.subplots(3, 3, figsize=(13, 3.2 * 3 * ALT / LARG + 1.2))
for lin, (nome, cfg) in enumerate(CONFIGS.items()):
    m = MASCARAS[nome]
    axs[lin, 0].imshow(IMG)
    if cfg.get("caixa") is not None:
        bx0, by0, bx1, by1 = caixa_px(cfg["caixa"])
        axs[lin, 0].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False,
                                        ec="yellow", lw=2))
    if cfg.get("pontos"):
        pp, rr = para_px(cfg["pontos"]), np.asarray(cfg["rotulos"])
        axs[lin, 0].scatter(pp[rr == 1, 0], pp[rr == 1, 1], c="lime", marker="*", s=150, ec="k", lw=.6)
        axs[lin, 0].scatter(pp[rr == 0, 0], pp[rr == 0, 1], c="red", marker="X", s=90, ec="k", lw=.6)
    axs[lin, 1].imshow(m, cmap="gray")
    axs[lin, 2].imshow(sobrepor(IMG, m))
    axs[lin, 0].set_ylabel(nome, fontsize=9)
    axs[lin, 1].set_xlabel(f"área = {100*m.mean():.2f}%", fontsize=8)
    for a in axs[lin]:
        a.set_xticks([]); a.set_yticks([])
for c, t in enumerate(["original + prompts (estrela: +, X: -)", "máscara binária", "sobreposição"]):
    axs[0, c].set_title(t)
fig.tight_layout(); fig.savefig("saida/03_comparativo.png", bbox_inches="tight"); plt.show()

---
# Etapa 4: avaliação no código

| Pergunta do enunciado | Medida |
|---|---|
| A máscara cobre a pluma? | `recall` contra a referência |
| Inclui o que não pertence a ela? | `precisão` e `invadiu ceu / nuvem_haze / edificacao (%)` |
| Deixa fumaça de fora? | `1 - recall` e o azul do mapa de erro |
| O resultado é estável? | `IoU` entre as três configurações |
| A borda corresponde a algo real? | separabilidade dentro contra fora (Bhattacharyya) |

As duas últimas linhas respondem à pergunta sobre iluminação, contraste e semelhança entre fumaça e
nuvem. IoU baixo entre configurações significa que três prompts razoáveis sobre a mesma imagem deram
respostas diferentes, ou seja, a resposta veio do prompt. Separabilidade baixa significa que a
máscara cortou uma região fotometricamente homogênea, sem fronteira na imagem para o contorno se
apoiar. A Etapa 4.2 reduz o contraste da cena e repete a predição com o mesmo prompt.

A máscara de referência é um polígono traçado à mão e aproximado. Na base da pluma não existe
fronteira objetiva entre fumaça e haze, então os valores de IoU e Dice servem para comparar as
configurações entre si. Essa incerteza do rótulo faz parte da resposta sobre por que a tarefa é
difícil.

In [ ]:
# ============================================================
# ETAPA 4 — Métricas, mapa de erro e efeito do contraste
# ============================================================
CINZA = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
SAT = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)[:, :, 1]


def bhattacharyya(a, b, bins=64):
    """0 = distribuições idênticas. 8 = valor de saturação, histogramas sem sobreposição."""
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    faixa = (float(min(a.min(), b.min())), float(max(a.max(), b.max())) + 1e-6)
    ha = np.histogram(a, bins, faixa)[0]; hb = np.histogram(b, bins, faixa)[0]
    ha = ha / (ha.sum() + 1e-12); hb = hb / (hb.sum() + 1e-12)
    return float(min(-np.log(np.sum(np.sqrt(ha * hb)) + 1e-12), 8.0))


def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 0.0


def metricas(m):
    n, _, _, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cont)
    d = {"área (%)": 100 * m.mean(), "componentes": int(n - 1),
         "compacidade": float(4 * np.pi * m.sum() / per ** 2) if per else 0.0}
    for nome, roi in ROIS.items():
        d[f"invadiu {nome} (%)"] = 100 * np.logical_and(m, roi).sum() / roi.sum()
    d["separab. luminância"] = bhattacharyya(CINZA[m].astype(float), CINZA[~m].astype(float))
    d["separab. saturação"] = bhattacharyya(SAT[m].astype(float), SAT[~m].astype(float))
    if REFERENCIA is not None:
        vp = np.logical_and(m, REFERENCIA).sum()
        d.update({"IoU": iou(m, REFERENCIA),
                  "Dice": float(2 * vp / (m.sum() + REFERENCIA.sum())) if m.sum() else 0.0,
                  "precisão": float(vp / m.sum()) if m.sum() else 0.0,
                  "recall": float(vp / REFERENCIA.sum())})
    return d


TABELA = pd.DataFrame({n: metricas(m) for n, m in MASCARAS.items()}).T.round(3)
display(TABELA)
TABELA.to_csv("saida/04_metricas.csv")

nomes = list(MASCARAS)
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        print(f"IoU({a[0]}, {b[0]}) = {iou(MASCARAS[a], MASCARAS[b]):.3f}")

# 4.1 mapa de erro da melhor configuração
if REFERENCIA is not None:
    melhor = TABELA["IoU"].idxmax()
    mm = MASCARAS[melhor]
    mapa = np.zeros((ALT, LARG, 3), np.uint8)
    mapa[np.logical_and(mm, REFERENCIA)] = (60, 200, 60)      # acerto
    mapa[np.logical_and(mm, ~REFERENCIA)] = (230, 70, 70)     # incluiu o que não é pluma
    mapa[np.logical_and(~mm, REFERENCIA)] = (70, 120, 240)    # deixou fumaça de fora
    fig, ax = plt.subplots(figsize=(9, 9 * ALT / LARG))
    ax.imshow((IMG * 0.45 + mapa * 0.55).astype(np.uint8)); ax.axis("off")
    ax.set_title(f"Mapa de erro, {melhor}. Verde: acerto. Vermelho: falso positivo. "
                 "Azul: falso negativo")
    fig.tight_layout(); fig.savefig("saida/04_erro.png", bbox_inches="tight"); plt.show()

# 4.2 mesma cena e mesmo prompt, com metade do contraste
base = MASCARAS["C: caixa + (+) + (-)"]
x = IMG.astype(np.float32) / 255
baixo = (np.clip((x - 0.5) * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)
cv2.imwrite("saida/_baixo_contraste.png", cv2.cvtColor(baixo, cv2.COLOR_RGB2BGR))
predictor.set_image("saida/_baixo_contraste.png")
m2 = prever(**CONFIGS["C: caixa + (+) + (-)"])
predictor.set_image(CAMINHO)
print(f"contraste x0.5: IoU vs original = {iou(m2, base):.3f}, "
      f"área {100*(m2.mean()-base.mean()):+.2f} p.p.")

fig, axs = plt.subplots(1, 2, figsize=(13, 13 * ALT / LARG / 2))
axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("original (C)")
axs[1].imshow(sobrepor(baixo, m2)); axs[1].set_title(f"metade do contraste, IoU = {iou(m2, base):.2f}")
for a in axs:
    a.axis("off")
fig.tight_layout(); fig.savefig("saida/04_contraste.png", bbox_inches="tight"); plt.show()

for nome, m in MASCARAS.items():
    cv2.imwrite(f"saida/mascara_{nome[0]}.png", (m * 255).astype(np.uint8))
    cv2.imwrite(f"saida/sobreposicao_{nome[0]}.png",
                cv2.cvtColor(sobrepor(IMG, m), cv2.COLOR_RGB2BGR))
print("gerado:", sorted(os.listdir("saida")))

---
# Etapa 5: conclusões

Confira os números da sua execução nas tabelas acima antes de entregar.

### A máscara cobre a pluma? Inclui o que não é? Deixa fumaça de fora?

Em A, o retângulo que contém a
pluma contém também muito céu, e o SAM devolve a região mais coerente dentro dele: a máscara vaza
para o céu, o que aparece em `invadiu ceu (%)` e na precisão, ou encolhe para o núcleo denso e perde
as bordas finas, o que derruba o recall. Em B, sem limite espacial, a máscara cresce por continuidade
de aparência e tende a engolir a faixa de haze do horizonte, que tem quase a mesma estatística da
base da pluma. Em C, os negativos em céu, haze, telhado, encosta e placa excluem distratores que
estão dentro da caixa; as colunas de invasão e a precisão dizem se funcionou nesta imagem.

### O que dificultou a segmentação

Fumaça e nuvem são ambas quase acromáticas, então cor não separa
as duas, e isso aparece na separabilidade de saturação. Fumaça também é um meio semitransparente:
perto da borda cada pixel é mistura de fumaça e fundo, não há gradiente forte para o contorno se
apoiar, e a borda acaba decidida pela caixa que desenhamos. A Etapa 4.2 mostra a mesma cena, com o
mesmo prompt, dando outra máscara só porque o contraste mudou, já que o SAM recalcula o embedding.
Na prática, fotos do mesmo incêndio em horários diferentes não são comparáveis sem normalização. A
placa de trânsito corta a pluma em duas, e sem um ponto negativo sobre ela a placa é absorvida pela
máscara. Por fim, o próprio rótulo é incerto: a referência traçada à mão hesita na base da pluma, e
nenhum IoU aqui deve ser lido como erro do modelo sem essa ressalva.

### O que eu faria a seguir

Refinamento iterativo, realimentando a máscara anterior como `masks=` no
prompt, com limpeza morfológica para remover ilhas isoladas, tomando o cuidado de não manter só o
maior componente conexo, o que descartaria a fumaça separada pela placa. Estimativa de transparência
em vez de máscara binária, que é o formato adequado a um meio semitransparente e a medir área de
pluma em série temporal. Referência de consenso, com duas ou três pessoas traçando a mesma pluma,
para saber qual é o teto de IoU possível nesta imagem. E repetir o protocolo em N imagens, com média
e desvio, em vez de um caso só.

## Entrega

Rode tudo e depois use `Salvar uma cópia no GitHub`, no menu `Arquivo`. O link a entregar é o do `.ipynb` no
GitHub. Rodar antes de salvar importa: o GitHub renderiza o notebook com as saídas embutidas, então o
professor vê as figuras e as tabelas sem executar nada.